In [1]:
## Load the RAG answer
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In production, we don't have original answer for real user. Still use LLM judge. The prompt has to judge only the question and the generated answer

In [3]:
## Define output format
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(description="Reasoning about the quality of the answer.")
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise"
    )

In [4]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [5]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [7]:
rec = answers[0]

In [8]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [9]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: late joining is allowed, but certificate eligibility requires submitting the project before submissions close. This is semantically equivalent.', score='good')

In [10]:
calc_price(usage)

{'input_cost': 0.00021975,
 'output_cost': 0.00022500000000000002,
 'total_cost': 0.00044475000000000005}

In [11]:
## Put into the function
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [12]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says the learner can still join, and that certificate eligibility requires submitting the project while submissions are still open. This preserves the key condition and meaning.', score='good')

In [13]:
## Running the judge on all answer
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )
    
    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [14]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=3) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/590 [00:00<?, ?it/s]

In [15]:
## Split results
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [16]:
## Create the dataframe
df_eval = pd.DataFrame(evaluations)

In [17]:
## Calculate the cost
calc_total_price(usages)

0.41912475000000027

In [18]:
## Check the results
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 559/590 = 94.75%


In [23]:
## Look at bad cases
df_eval[df_eval["score"] == "bad"][['question', 'reasoning']].head()

,question,reasoning
1,Is it too late to start the course if I missed the beginning?,"The ground truth says it is not too late to start the course, but adds an important condition for getting a certificate: you must submit the project while submissions are still being accepted. The AI answer correctly says it's not too late and you can start whenever you want, but it omits the certificate/submission condition. Since that is a key part of the original answer, the responses are not fully equivalent."
2,Can I enroll after the course has already started?,"The AI answer does not convey the ground truth. The correct answer is yes, enrollment is possible after the course has started, with the condition that certificate seekers must submit the project while submissions are still open. Saying 'I don't know' misses this key information and is incorrect."
4,What’s the deadline for getting the certificate if I’m joining the course late?,The ground truth says late joiners can still get a certificate if they submit their project while submissions are still being accepted. The AI answer gives no useful information and misses the key condition entirely.
9,"If I filled out the form, does that mean I’m officially accepted already?","The ground truth says the student is already accepted and that registration is only for gauging interest, not required for acceptance. The AI answer fails to convey this and instead says 'I don't know,' which is incorrect and misses the key point."
32,"Is homework required for passing the course, or is it only there to help with practice?","The AI answer captures the main point that homework is not required and is mainly for practice/leaderboard. However, it introduces 'required peer reviews' as important for the certificate, which is not mentioned in the ground truth. The original answer only states that passing the Capstone project is required to get the certificate; adding another requirement changes the meaning. Therefore it is not semantically equivalent."


In [20]:
## Save judged answer
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)

In [22]:
pd.set_option('display.max_colwidth', None)